# Auditoría de experimentos — nlp-lab2-sentiment140

Este notebook se conecta al MLflow Tracking Server (variable `MLFLOW_TRACKING_URI`) y reconstruye, directamente desde MLflow (no desde copias locales, como exige la Sección 6), el run de protocolo, la tabla de runs experimentales/finales, y las contribuciones por integrante. Se usa como apoyo para la sustentación (Sección 8): no forma parte del contrato de caja negra de la API.

In [ ]:
import os
import mlflow
from mlflow.tracking import MlflowClient

EXPERIMENT_NAME = "nlp-lab2-sentiment140"

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
client = MlflowClient()

experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
assert experiment is not None, f"No existe el experiment '{EXPERIMENT_NAME}' en este Tracking Server."
experiment.experiment_id

## Runs presentados

Solo se consideran "presentados" los runs con tag `lab_run_type` en `{protocol, experiment, final}` (A.1); los runs exploratorios sin ese tag se ignoran.

In [ ]:
all_runs = client.search_runs(experiment_ids=[experiment.experiment_id], max_results=50_000)
presented_runs = [r for r in all_runs if r.data.tags.get("lab_run_type") in {"protocol", "experiment", "final"}]
len(presented_runs)

## Run de protocolo

Debe existir exactamente uno con `lab_run_type=protocol` (A.2).

In [ ]:
protocol_runs = [r for r in presented_runs if r.data.tags.get("lab_run_type") == "protocol"]
assert len(protocol_runs) == 1, f"Se esperaba exactamente 1 run de protocolo, hay {len(protocol_runs)}."
protocol_run = protocol_runs[0]
protocol_run.info.run_id, dict(protocol_run.data.params)

## Tabla de runs experimentales (T0, B0, P_*, R_*, C_*, EXTRA, ABLATION)

In [ ]:
import pandas as pd

experiment_runs = [r for r in presented_runs if r.data.tags.get("lab_run_type") == "experiment"]

rows = []
for r in experiment_runs:
    rows.append({
        "run_id": r.info.run_id,
        "lab_experiment_id": r.data.tags.get("lab_experiment_id"),
        "lab_stage": r.data.tags.get("lab_stage"),
        "lab_member_id": r.data.tags.get("lab_member_id"),
        "lab_configuration_id": r.data.tags.get("lab_configuration_id"),
        "macro_f1_mean": r.data.metrics.get("macro_f1_mean"),
        "macro_f1_std": r.data.metrics.get("macro_f1_std"),
        "macro_f1_delta": r.data.metrics.get("macro_f1_delta"),
    })

experiments_df = pd.DataFrame(rows).sort_values("run_id").reset_index(drop=True)
experiments_df

## Run final y modelo registrado (`sentiment140@champion`)

In [ ]:
final_runs = [r for r in presented_runs if r.data.tags.get("lab_run_type") == "final"]
for r in final_runs:
    print(r.info.run_id, r.data.tags.get("lab_selected_experiment_run_id"), r.data.metrics.get("test_macro_f1"))

champion = client.get_model_version_by_alias("sentiment140", "champion")
champion.version, champion.run_id

## Contribuciones por integrante

El mínimo exige `valid_configurations >= 3` y al menos 2 etapas distintas (Sección 3); T0/B0 no cuentan.

In [ ]:
by_member = {}
for r in experiment_runs:
    member_id = r.data.tags.get("lab_member_id")
    experiment_id = r.data.tags.get("lab_experiment_id")
    if experiment_id in ("T0", "B0") or member_id is None:
        continue
    entry = by_member.setdefault(member_id, {"configuration_ids": set(), "stages": set()})
    config_id = r.data.tags.get("lab_configuration_id")
    stage = r.data.tags.get("lab_stage")
    if config_id:
        entry["configuration_ids"].add(config_id)
    if stage:
        entry["stages"].add(stage)

contributions_df = pd.DataFrame([
    {
        "member_id": member_id,
        "valid_configurations": len(entry["configuration_ids"]),
        "stages": sorted(entry["stages"]),
        "cumple_minimo": len(entry["configuration_ids"]) >= 3 and len(entry["stages"]) >= 2,
    }
    for member_id, entry in by_member.items()
]).sort_values("member_id").reset_index(drop=True)

contributions_df